Aim of this script: pick up nb4's stop-level cancellation dataset, collapse it into a per-train (`journeys`) table, build the final operator / route-type / year summary, and export it as an Excel workbook in `summary_stats/`.

In [1]:
import pandas as pd
import os

In [2]:
data_selection = "french"

intermediate_outputs_dir = "intermediate_outputs"
data_path = f"{intermediate_outputs_dir}/data_chuuchuu_{data_selection}_cancellations.parquet"

# try:
#     data_chuuchuu.head()
# except NameError:
#     data_chuuchuu = pd.read_parquet(data_path)
data_chuuchuu = pd.read_parquet(data_path)

data_chuuchuu.shape

(7035704, 38)

In [3]:
data_chuuchuu["normalized_operator"].value_counts(dropna=False)

normalized_operator
SNCF                                       6420840
SNCF VOYAGEURS                              337966
Eurostar                                    153926
Deutsche Bahn                                62093
Trenitalia                                   24845
SNCF Voyageurs LO                            21041
OCEdefault                                   12469
Conseil Régional Auvergne - Rhône-Alpes       1727
SNCF Voyageurs EA                              718
SNCF Voyageurs SA                               79
Name: count, dtype: int64

### Building a per-train summary for the operator / route-type report

nb4 treats each row as one stop. This summary needs one row per train instead, so we collapse each `journey_id` down to a single record.

We reuse the same exclusions as nb2/nb3: a journey with a repeated stop id (`is_ambiguous_trip`) can't be ranked into depart/intermediate/terminus, and a `journey_verificator` collision flagged `likely_true_duplicate` is almost certainly the same physical train reported twice by two agencies -- keeping both would double-count it.

**Cancellation** is judged across every stop of the journey (using `arrivalCancelled_resolved` from nb4):
- a journey is only classified if every one of its stops has a known cancellation status -- a single unresolved stop could be hiding either outcome, so we don't guess
- **full cancellation**: all stops cancelled
- **partial cancellation**: some, but not all, stops cancelled

**Terminus outcome** (`cancelled_terminus`) looks specifically at the journey's `terminus` stop: was it cancelled, not cancelled, or unresolved (no terminus row could be identified for that journey)? `arrived_at_terminus` is true only when the terminus was resolved and confirmed not cancelled. Delay (>5min / >15min) is measured on that same terminus row's `arrivalDelay`, in line with the country-punctuality check above -- a cancelled or unresolved terminus has no delay to measure, so it's excluded from both delay counts rather than counted as on-time.

In [4]:
# this summary needs one row per train, not one row per stop -- exclude the same
# unreliable journeys nb2/nb3 already flagged (see markdown above)
clean = ~data_chuuchuu["is_ambiguous_trip"] & (data_chuuchuu["cross_agency_duplicate_confidence"] != "likely_true_duplicate")
print(f"{(~clean).sum()} rows excluded as an ambiguous trip or a likely cross-agency duplicate")

clean_data = data_chuuchuu[clean].copy()
clean_data["year"] = pd.to_datetime(clean_data["date"]).dt.year

46 rows excluded as an ambiguous trip or a likely cross-agency duplicate


In [5]:
stop_counts = clean_data.groupby("journey_id").agg(
    n_stops=("arrivalCancelled_resolved", "size"),
    n_known_cancellation=("arrivalCancelled_resolved", lambda s: s.notna().sum()),
    n_cancelled_stops=("arrivalCancelled_resolved", lambda s: (s == "t").sum()),
)

# only classify a journey's cancellation if every one of its stops has a known status
stop_counts["cancellation_reliable"] = stop_counts["n_known_cancellation"] == stop_counts["n_stops"]
stop_counts["is_fully_cancelled"] = stop_counts["cancellation_reliable"] & (stop_counts["n_cancelled_stops"] == stop_counts["n_stops"])
stop_counts["is_partially_cancelled"] = (
    stop_counts["cancellation_reliable"]
    & (stop_counts["n_cancelled_stops"] > 0)
    & (stop_counts["n_cancelled_stops"] < stop_counts["n_stops"])
)

print(f"{(~stop_counts['cancellation_reliable']).sum()} of {len(stop_counts)} journeys have no reliable cancellation status (excluded from full/partial counts)")
stop_counts[["is_fully_cancelled", "is_partially_cancelled"]].sum()

0 of 857791 journeys have no reliable cancellation status (excluded from full/partial counts)


is_fully_cancelled        11822
is_partially_cancelled     9714
dtype: int64

In [6]:
# journey attributes (operator, route type, year) are constant across a journey's stops by
# construction (journey_id == agency + routeType + routeNumber + date), so any stop can represent them
journey_attrs = clean_data.drop_duplicates("journey_id").set_index("journey_id")[
    ["agency", "normalized_operator", "routeType", "year", "journey_type"]
].rename(columns={"normalized_operator": "operator"})

terminus_rows = clean_data.loc[clean_data["depart_terminus"] == "terminus"].set_index("journey_id")
cancelled_terminus = terminus_rows["arrivalCancelled_resolved"].map({"t": True, "f": False})  # NaN where unresolved

journeys = journey_attrs.join(stop_counts)
journeys["cancelled_terminus"] = journeys.index.map(cancelled_terminus)
journeys["terminus_arrival_delay"] = journeys.index.map(terminus_rows["arrivalDelay"])

# arrived at terminus: the terminus stop was identified AND confirmed not cancelled (NaN --
# no terminus row could be resolved for that journey -- correctly evaluates to False here)
journeys["arrived_at_terminus"] = journeys["cancelled_terminus"] == False

DELAY_5MIN_SECONDS = 5 * 60
DELAY_15MIN_SECONDS = 15 * 60
journeys["delayed_5min"] = journeys["arrived_at_terminus"] & (journeys["terminus_arrival_delay"] > DELAY_5MIN_SECONDS)
journeys["delayed_15min"] = journeys["arrived_at_terminus"] & (journeys["terminus_arrival_delay"] > DELAY_15MIN_SECONDS)

# the groupby below needs journey_id as a regular column, not the index
journeys = journeys.reset_index()

print(f"{len(journeys)} journeys total")
print(f"{journeys['arrived_at_terminus'].sum()} arrived at terminus")
print(f"{journeys['delayed_5min'].sum()} delayed >5min, {journeys['delayed_15min'].sum()} delayed >15min (at terminus)")

journeys.head()

857791 journeys total
843039 arrived at terminus
95848 delayed >5min, 49188 delayed >15min (at terminus)


,journey_id,agency,operator,routeType,year,journey_type,n_stops,n_known_cancellation,n_cancelled_stops,cancellation_reliable,is_fully_cancelled,is_partially_cancelled,cancelled_terminus,terminus_arrival_delay,arrived_at_terminus,delayed_5min,delayed_15min
0,FR_INTERCITES_3604_2025-01-01,FR,SNCF,INTERCITES,2025,domestic,7,7,0,True,False,False,False,0.0,True,False,False
1,FR_TGV INOUI_8330_2025-01-01,FR,SNCF,TGV INOUI,2025,domestic,5,5,0,True,False,False,False,0.0,True,False,False
2,FR_TGV INOUI_2501_2025-01-01,FR,SNCF,TGV INOUI,2025,domestic,3,3,0,True,False,False,False,0.0,True,False,False
3,FR_INTERCITES_5950_2025-01-01,FR,SNCF,INTERCITES,2025,domestic,6,6,0,True,False,False,False,0.0,True,False,False
4,FR_TGV INOUI_2714_2025-01-01,FR,SNCF,TGV INOUI,2025,domestic,5,5,0,True,False,False,False,0.0,True,False,False


### Aggregating to Year / operator / Route type

Each row of `journeys` (built above) is already one train. `operator` is null wherever nb3 couldn't resolve one -- rather than silently dropping those trains from the groupby, they're kept under an explicit `UNKNOWN_OPERATOR` label so the totals in the summary still add up to the full `journeys` count.

Column definitions (see the per-train summary section above for how each is derived):
- **N total train** -- every train in scope, regardless of cancellation/delay/operator status
- **N cancelled train (full cancellation)** -- every stop of the train was cancelled
- **N cancelled train (partial cancellation)** -- some, but not all, stops were cancelled
- **N cancelled train (full + partial)** -- either of the above
- **N train delay (> 5 min) / (> 15 min)** -- arrived at the terminus, but later than the threshold
- **N train arriving at terminus** -- the terminus stop was reached (not cancelled), independent of delay

In [7]:
journeys["operator"] = journeys["operator"].fillna("UNKNOWN_OPERATOR")
print(f"{(journeys['operator'] == 'UNKNOWN_OPERATOR').mean() * 100:.2f}% of trains have no resolved operator")

summary = journeys.groupby(["year", "operator", "routeType"], dropna=False).agg(
    n_total_train=("journey_id", "size"),
    n_cancelled_full=("is_fully_cancelled", "sum"),
    n_cancelled_partial=("is_partially_cancelled", "sum"),
    n_delay_5min=("delayed_5min", "sum"),
    n_delay_15min=("delayed_15min", "sum"),
    n_arriving_terminus=("arrived_at_terminus", "sum"),
).reset_index()

summary["n_cancelled_full_or_partial"] = summary["n_cancelled_full"] + summary["n_cancelled_partial"]

assert summary["n_total_train"].sum() == len(journeys), "grouped total doesn't match the journey count -- a group key must be dropping nulls"

summary.head()

0.00% of trains have no resolved operator


,year,operator,routeType,n_total_train,n_cancelled_full,n_cancelled_partial,n_delay_5min,n_delay_15min,n_arriving_terminus,n_cancelled_full_or_partial
0,2025,Conseil Régional Auvergne - Rhône-Alpes,CAR,56,0,0,0,0,56,0
1,2025,Conseil Régional Auvergne - Rhône-Alpes,CTE,11,0,0,8,0,11,0
2,2025,Conseil Régional Auvergne - Rhône-Alpes,Car,176,0,29,0,0,176,29
3,2025,Deutsche Bahn,ICE,8289,13,207,3776,1886,8218,220
4,2025,Eurostar,EST,42687,1775,2646,10548,6311,40398,4421


### Exporting the summary as Excel

In [8]:
summary = summary.rename(columns={
    "year": "Year",
    "routeType": "Route type",
    "n_total_train": "N total train",
    "n_cancelled_full": "N cancelled train (full cancellation)",
    "n_cancelled_partial": "N cancelled train (partial cancellation)",
    "n_cancelled_full_or_partial": "N cancelled train (full + partial)",
    "n_delay_5min": "N train delay (> 5 min)",
    "n_delay_15min": "N train delay (> 15 min)",
    "n_arriving_terminus": "N train arriving at terminus",
})

column_order = [
    "Year", "operator", "Route type",
    "N total train",
    "N cancelled train (full cancellation)",
    "N cancelled train (partial cancellation)",
    "N cancelled train (full + partial)",
    "N train delay (> 5 min)",
    "N train delay (> 15 min)",
    "N train arriving at terminus",
]
summary = summary[column_order].sort_values(["Year", "operator", "Route type"])
summary

,Year,operator,Route type,N total train,N cancelled train (full cancellation),N cancelled train (partial cancellation),N cancelled train (full + partial),N train delay (> 5 min),N train delay (> 15 min),N train arriving at terminus
0,2025,Conseil Régional Auvergne - Rhône-Alpes,CAR,56,0,0,0,0,0,56
1,2025,Conseil Régional Auvergne - Rhône-Alpes,CTE,11,0,0,0,8,0,11
2,2025,Conseil Régional Auvergne - Rhône-Alpes,Car,176,0,29,29,0,0,176
3,2025,Deutsche Bahn,ICE,8289,13,207,220,3776,1886,8218
4,2025,Eurostar,EST,42687,1775,2646,4421,10548,6311,40398
5,2025,OCEdefault,CTE,1537,0,0,0,51,23,1537
6,2025,SNCF,CAR TER,13630,0,0,0,1211,233,13630
7,2025,SNCF,IC,3423,37,31,68,730,461,3375
8,2025,SNCF,ICN,504,46,12,58,26,18,453
9,2025,SNCF,INTERCITES,18995,218,226,444,4208,2723,18687


In [9]:
export_summary = input("Export summary to Excel? (y/n): ")

if export_summary.lower() == "y":
    summary_stats_dir = "summary_stats"
    os.makedirs(summary_stats_dir, exist_ok=True)

    summary_path = f"{summary_stats_dir}/chuuchuu_summary_{data_selection}.xlsx"
    summary.to_excel(summary_path, index=False)
    print(f"Saved to {summary_path}")

Saved to summary_stats/chuuchuu_summary_french.xlsx
